In [1]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster()
client = cluster.get_client()

/home/ilya/Projects/WMSs/wms-comparison/dask/dask-venv/lib/python3.10/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37441 instead
  warnings.warn(


In [2]:
cluster.dashboard_link

'http://127.0.0.1:37441/status'

In [ ]:
from dask import delayed, compute
from pathlib import Path
import time


# Specify the directory path
directory = Path("image_processing_results")

# Create the folder (and parent folders if missing)
directory.mkdir(parents=True, exist_ok=True)

@delayed
def crop_task_wrapper(*args):
    import subprocess
    subprocess.run(["python3", "../image_processing/crop.py", *map(str, args)], check=True)
    return args[-1]

@delayed
def sobel_task_wrapper(infile, *args):
    import subprocess
    subprocess.run(["python", "../image_processing/sobel.py", infile, *map(str, args)], check=True)
    return args[-1]

@delayed
def binarize_task_wrapper(infile, *args):
    import subprocess
    subprocess.run(["python", "../image_processing/binarize.py", infile, *map(str, args)], check=True)
    return args[-1]

@delayed
def height_task_wrapper(infile, *args):
    import subprocess
    subprocess.run(["python", "../image_processing/droplet_height.py", infile, *map(str, args)], check=True)

def get_next_file_path(prevFile: str, operationName: str):
    fileSplit = prevFile.split("/")
    return fileSplit[0] + "/" + operationName + fileSplit[1]  

directory = Path('/home/ilya/Projects/WMSs/image_processing_input_data')
start_time = time.perf_counter()
print(f"Время старта: {start_time:.4f} секунд")
# Код для измерения
for file in directory.iterdir():
    if file.is_file():
        cropped_file = crop_task_wrapper(
            file, 464, 193, 169, 154, "image_processing_results/crop_" + file.name
        )
        sobel_file = sobel_task_wrapper(cropped_file, get_next_file_path(cropped_file, "sobel_"))
        binarize_file = binarize_task_wrapper(sobel_file, 50, get_next_file_path(sobel_file, "binarize_"))
        final_result = height_task_wrapper(binarize_file, get_next_file_path(binarize_file, "height_value_"), get_next_file_path(binarize_file, "height_file_"))

        compute(final_result)
end_time = time.perf_counter()
print(f"Время выполнения: {end_time - start_time:.4f} секунд")

Время старта: 56969.5045 секунд
Время выполнения: 268.4803 секунд
